# Assignment 2 - Multilayer Perceptron Grid Search

In this assignment, we will be building and evaluating machine learning models (specifically, neural networks) to perform image classification using PyTorch on the CIFAR-100 dataset.

We'll start with one of the most basic neural network architectures, a **Multilayer Perceptron (MLP)**, also known as a feedforward network. Unlike the MNIST dataset used in our tutorials, CIFAR-100 consists of 32x32 color images across 100 classes (grouped into 20 superclasses/coarse labels). Due to the complexity of color image data, MLPs are expected to perform modestly, but they provide a perfect foundation for understanding hyperparameter optimization.

Our primary objective is to perform a comprehensive **manual grid search** over 64 different hyperparameter combinations to identify the most effective configuration for this architecture.

### Data Processing and Setup

Let's start by importing all the modules we'll need. The main ones we need to import are:
- `torch` for general PyTorch functionality
- `torch.nn` for neural network based functions and layers
- `torch.optim` for our optimizers (Adam and Adagrad) which will update the parameters of our neural network
- `torch.utils.data` for handling the dataset and batching
- `numpy` for data manipulation and preprocessing
- `pickle` for loading the local dataset files
- `itertools` for generating the hyperparameter grid

## 1. Imports and Device Setup

To ensure we can train our models efficiently, we first check if a GPU (CUDA) is available. Using a GPU significantly accelerates the training process for neural networks compared to a standard CPU.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import itertools
import csv
import os
import pickle

# Set device to GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Defining the Hyperparameter Grid

Hyperparameters are the "knobs" we turn to control the learning process. In this assignment, we will explore 64 unique combinations derived from:
- **Hidden Units:** The width of the hidden layers (120 or 240).
- **Hidden Activations:** Non-linear functions applied after hidden layers (ReLU or Sigmoid).
- **Output Activation:** The final layer's activation (Softmax or Sigmoid).
- **Loss Functions:** The error metric to minimize (Mean Squared Error or Categorical Cross-Entropy).
- **Optimizers:** The algorithm used to update weights (Adam or Adagrad).
- **Batch Size:** The number of samples processed before updating weights (1000 or 2000).

In [2]:
param_dict = {
    'units': [120, 240],
    'hidden_activations': ['relu', 'sigmoid'],
    'activation': ['softmax', 'sigmoid'],
    'loss': ['mse', 'categorical_crossentropy'],
    'optimizer': ['adam', 'adagrad'],
    'batch_size': [1000, 2000]
}

## 3. Data Loading and Normalization

The CIFAR-100 dataset is provided as local binary "pickle" files. We need to load these files, extract the image data and labels, and normalize the features.

**Normalization:** We scale the pixel values from their original range [0, 255] to [0, 1] by dividing by 255.0. This helps the model train faster and avoid local minima by ensuring all input features are on a similar scale.

**Labels:** For `Categorical Cross-Entropy` loss, PyTorch expects class indices. For `MSE` loss, we require one-hot encoded labels, where each target is represented as a vector of zeros with a one at the index of the correct class.

In [7]:
# Load the provided CIFAR-100 pickle files
try:
    with open('./train', 'rb') as f:
        train_dict = pickle.load(f, encoding='bytes')
    with open('./test', 'rb') as f:
        test_dict = pickle.load(f, encoding='bytes')

    # Normalize the image data to the range [0, 1]
    x_train = train_dict[b'data'].astype('float32') / 255.0
    y_train = np.array(train_dict[b'coarse_labels'])
    x_test = test_dict[b'data'].astype('float32') / 255.0
    y_test = np.array(test_dict[b'coarse_labels'])

    print('x_train shape:', x_train.shape)
    print(f"{x_train.shape[0]} train samples")
    print(f"{x_test.shape[0]} test samples")

    # Prepare One-hot encoding for MSE configurations
    num_classes = len(np.unique(y_train))
    print('num_classes:', num_classes)
    y_train_oh = np.eye(num_classes, dtype='float32')[y_train]
    y_test_oh = np.eye(num_classes, dtype='float32')[y_test]
except FileNotFoundError:
    print("Error: 'train' or 'test' files not found. Please upload them to the current directory.")

x_train shape: (50000, 3072)
50000 train samples
10000 test samples
num_classes: 20


## 4. Defining the Model and Training Loop

Our MLP architecture must include at least **5 hidden layers**. We define a function `my_model` that dynamically constructs this network based on the current hyperparameter permutation.

### Implementation Nuances:
- **Hidden Layers:** We use `nn.Linear` layers followed by the selected hidden activation function (ReLU or Sigmoid).
- **Loss Function Requirements:** 
  - `nn.CrossEntropyLoss` combines `LogSoftmax` and `NLLLoss` in one single class. Therefore, we **must not** apply an activation function to the output layer when using this loss.
  - `nn.MSELoss` requires the output to be activated manually (using Softmax or Sigmoid) and expects the target to be one-hot encoded.
- **Optimization:** We train for exactly **200 epochs** per combination to ensure consistency across the grid search.

In [ ]:
def make_hidden_activation(name):
    return nn.ReLU() if name == 'relu' else nn.Sigmoid()

def my_model(x_train, y_train_oh, y_train_idx, x_val, y_val_oh, y_val_idx, params):
    use_ce = (params['loss'] == 'categorical_crossentropy')
    out_act_fn = nn.Sigmoid() if params['activation'] == 'sigmoid' else nn.Softmax(dim=1)

    # Build an MLP with 5 hidden Linear layers as required by the assignment
    layers = [nn.Linear(3072, params['units']), make_hidden_activation(params['hidden_activations'])]

    # Add the remaining 4 hidden layers (total 5 hidden)
    for _ in range(4):
        layers.append(nn.Linear(params['units'], params['units']))
        layers.append(make_hidden_activation(params['hidden_activations']))

    # Final Output Layer
    layers.append(nn.Linear(params['units'], num_classes))
    model = nn.Sequential(*layers).to(device)

    # Select Optimizer
    opt = (optim.Adam(model.parameters()) 
           if params['optimizer'] == 'adam' 
           else optim.Adagrad(model.parameters()))

    # Select Loss Function
    criterion = nn.CrossEntropyLoss() if use_ce else nn.MSELoss()

    # Prepare Dataset based on Loss Function requirements
    if use_ce:
        dataset = TensorDataset(torch.tensor(x_train, dtype=torch.float32), 
                                torch.tensor(y_train_idx, dtype=torch.long))
    else:
        dataset = TensorDataset(torch.tensor(x_train, dtype=torch.float32), 
                                torch.tensor(y_train_oh, dtype=torch.float32))

    loader = DataLoader(dataset, batch_size=params['batch_size'], shuffle=True)

    # Training Loop
    model.train()
    for epoch in range(200):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            opt.zero_grad()
            output = model(X_batch)

            # Logic: CrossEntropy expects logits; MSE expects activated outputs
            loss = criterion(output, y_batch) if use_ce else criterion(out_act_fn(output), y_batch)
            
            loss.backward()
            opt.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        train_logits = model(torch.tensor(x_train, dtype=torch.float32).to(device))
        val_logits = model(torch.tensor(x_val, dtype=torch.float32).to(device))

        train_out = train_logits if use_ce else out_act_fn(train_logits)
        val_out = val_logits if use_ce else out_act_fn(val_logits)

        train_acc = (train_out.argmax(dim=1) == torch.tensor(y_train_idx).to(device)).float().mean().item()
        val_acc = (val_out.argmax(dim=1) == torch.tensor(y_val_idx).to(device)).float().mean().item()

    return {'accuracy': round(train_acc, 4), 'val_accuracy': round(val_acc, 4)}

## 5. Executing the Grid Search

We now iterate through the Cartesian product of all hyperparameter lists. This process will systematically train and evaluate 64 different models. We track our progress and log the validation accuracy for each combination to identify the best performer.

In [ ]:
keys = list(param_dict.keys())
combinations = list(itertools.product(*param_dict.values()))
results = []

print(f"Starting grid search: {len(combinations)} total combinations.")

for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    metrics = my_model(x_train, y_train_oh, y_train, x_test, y_test_oh, y_test, params)
    results.append({**params, **metrics})
    
    if (i + 1) % 5 == 0 or (i + 1) == len(combinations):
        print(f"Progress: {i+1}/{len(combinations)} | {params} => val_acc: {metrics['val_accuracy']}")

Starting grid search: 64 total combinations.
Progress: 5/64 | {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'categorical_crossentropy', 'optimizer': 'adam', 'batch_size': 1000} => val_acc: 0.1832
Progress: 10/64 | {'units': 120, 'hidden_activations': 'relu', 'activation': 'sigmoid', 'loss': 'mse', 'optimizer': 'adam', 'batch_size': 2000} => val_acc: 0.05
Progress: 15/64 | {'units': 120, 'hidden_activations': 'relu', 'activation': 'sigmoid', 'loss': 'categorical_crossentropy', 'optimizer': 'adagrad', 'batch_size': 1000} => val_acc: 0.0957
Progress: 20/64 | {'units': 120, 'hidden_activations': 'sigmoid', 'activation': 'softmax', 'loss': 'mse', 'optimizer': 'adagrad', 'batch_size': 2000} => val_acc: 0.0548
Progress: 25/64 | {'units': 120, 'hidden_activations': 'sigmoid', 'activation': 'sigmoid', 'loss': 'mse', 'optimizer': 'adam', 'batch_size': 1000} => val_acc: 0.05
Progress: 30/64 | {'units': 120, 'hidden_activations': 'sigmoid', 'activation': 'sigmoid', 

## 6. Saving and Analyzing Results

Once the grid search is complete, we persist the results into a CSV file. This allows for long-term storage and easier analysis using tools like Excel or Pandas. We also display the top 10 configurations based on validation accuracy to conclude our study.

In [6]:
import pandas as pd

# Create output directory
os.makedirs('grid_search_output', exist_ok=True)

# Save to CSV
csv_path = 'grid_search_output/results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print(f"Results saved to {csv_path}")

# Display top 10 results
df = pd.DataFrame(results)
print("\nTop 10 configurations by Validation Accuracy:")
print(df.sort_values('val_accuracy', ascending=False).head(10).to_string(index=False))

Results saved to grid_search_output/results.csv

Top 10 configurations by Validation Accuracy:
 units hidden_activations activation                     loss optimizer  batch_size  accuracy  val_accuracy
   240               relu    sigmoid categorical_crossentropy      adam        1000    0.2032        0.1999
   240               relu    softmax categorical_crossentropy      adam        1000    0.2064        0.1978
   120               relu    softmax categorical_crossentropy      adam        1000    0.1872        0.1832
   120               relu    softmax                      mse      adam        1000    0.1866        0.1797
   120               relu    sigmoid categorical_crossentropy      adam        1000    0.1815        0.1775
   120               relu    softmax                      mse      adam        2000    0.1841        0.1742
   240               relu    sigmoid categorical_crossentropy      adam        2000    0.1756        0.1735
   240               relu    softmax cate